# 🧠 Decovo v2: 1-Photon Miniscope CNMF-E Pipeline (AVI & Multi-Page TIFF)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/GabrielMQueiroz/CalciumImage/blob/main/Decovo/notebooks/Decovo_Colab.ipynb)

**How to use (no coding required):**
1. Put your session recording (either a single multi-page `.tif`/`.tiff` stack OR `msCam*.avi` chunks) inside a folder on your **Google Drive**.
2. Run **Step 1** (CondaColab) — wait ~30 seconds for the runtime to restart automatically.
3. Run **Step 2** & **Step 3** to install CaImAn + `decovo` and mount your Google Drive.
4. Fill in your Drive folder path and frame rate in **Step 4**, verify the metadata check in **Step 5**, and run **Step 6 (Stage A)**.
5. Use the sliders in **Step 7 (Stage B)** to tune thresholds and extract neurons in minutes (without redoing motion correction!), then download `analysis_results.hdf5` for the **CleanMiniscope Cell Curator GUI**.

> 💡 **Note on Colab Runtime:** A standard **CPU runtime** is recommended. CNMF-E 1-photon initialization and OASIS deconvolution are CPU-bound.

In [ ]:
# @title 1. Install CondaColab (Run once — runtime will restart automatically)
try:
    import condacolab
    condacolab.check()
    print('✅ CondaColab is already active!')
except Exception:
    !pip install -q condacolab
    import condacolab
    condacolab.install()

In [ ]:
# @title 2. Install CaImAn & Decovo (~3-5 minutes on first run)
import os, shutil, subprocess
from pathlib import Path
import condacolab
condacolab.check()

try:
    import caiman
    print(f'✅ CaImAn {caiman.__version__} already installed.')
except ImportError:
    print('⏳ Installing CaImAn via mamba (~3-5 min)...')
    !mamba install -y -c conda-forge caiman tifffile pyyaml ipywidgets psutil

print('⏳ Installing Decovo package...')
ret = subprocess.call([
    'pip', 'install', '-q', '--no-deps',
    'git+https://github.com/GabrielMQueiroz/CalciumImage.git#subdirectory=Decovo'
])
if ret != 0:
    for fallback_dir in ('/content/Decovo', '/content/drive/MyDrive/Decovo', '/content/drive/MyDrive/CalciumImage/Decovo'):
        if Path(fallback_dir).exists():
            print(f'ℹ️ Installing Decovo from local/Drive path: {fallback_dir}')
            subprocess.check_call(['pip', 'install', '-q', '--no-deps', '-e', fallback_dir])
            break

from google.colab import output
output.enable_custom_widget_manager()
import decovo
print(f'✅ Decovo v{decovo.__version__} ready!')

In [ ]:
# @title 3. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# @title 4. Configure Session Parameters
input_folder = '/content/drive/MyDrive/CalciumData/Session1' # @param {type:"string"}
frame_rate = 20.0 # @param {type:"number"}
use_metadata_fps_if_present = True # @param {type:"boolean"}
spatial_downsample = 0.6 # @param {type:"slider", min:0.2, max:1.0, step:0.05}
gSig = 3.0 # @param {type:"number"}
preset = 'miniscope_1p_default' # @param ["miniscope_1p_default"]
start_fresh = False # @param {type:"boolean"}
keep_memmap_on_drive = False # @param {type:"boolean"}

effective_frame_rate = float(frame_rate)
print(f'Configured input_folder = {input_folder}')
print(f'Configured frame_rate = {frame_rate} Hz (use_metadata_fps_if_present={use_metadata_fps_if_present}), spatial_downsample = {spatial_downsample}, gSig = {gSig} (gSiz = {int(4*gSig+1)})')

In [ ]:
# @title 5. Check Input Recording & Verify Frame Rate Metadata
# @markdown Reads TIFF/AVI headers, reports detected frame rate from metadata (if present), and checks RAM requirements.
# @markdown **Please check the reported Metadata Frame Rate below by hand and adjust `frame_rate` (or uncheck `use_metadata_fps_if_present`) in Step 4 if needed.**
import os, shutil
from pathlib import Path
from decovo.notebook_ui import inspect_input_banner

spec = inspect_input_banner(
    input_folder,
    user_frame_rate=frame_rate,
    spatial_downsample=spatial_downsample,
)
if use_metadata_fps_if_present and spec.fps_from_metadata is not None and spec.fps_from_metadata > 0:
    effective_frame_rate = float(spec.fps_from_metadata)
    print(f'👉 Active frame_rate set from file metadata: {effective_frame_rate:.2f} Hz (please verify by hand; uncheck use_metadata_fps_if_present in Step 4 to override manually).')
else:
    effective_frame_rate = float(frame_rate)
    print(f'👉 Active frame_rate set from your manual form value: {effective_frame_rate:.2f} Hz')

In [ ]:
# @title 6. Stage A: Preprocess, Motion Correction & Summary Images (Resumable)
# @markdown Copies input to fast local SSD (`/content/decovo_work`), restores any existing checkpoints from Google Drive, runs Stage A, and mirrors checkpoints back to Drive.
import hashlib, os, shutil
from pathlib import Path
from decovo import discover_inputs, load_config, preprocess, extract

spec = discover_inputs(input_folder)
if use_metadata_fps_if_present and spec.fps_from_metadata is not None and spec.fps_from_metadata > 0:
    effective_frame_rate = float(spec.fps_from_metadata)
else:
    effective_frame_rate = float(frame_rate)

in_path = Path(input_folder).resolve()
drive_base_dir = in_path.parent if in_path.is_file() else in_path
drive_results_dir = drive_base_dir / 'results_decovo'
drive_results_dir.mkdir(parents=True, exist_ok=True)

parent_prefix = f'{drive_base_dir.parent.name}_' if drive_base_dir.parent.name else ''
path_hash = hashlib.sha256(str(drive_base_dir).encode('utf-8')).hexdigest()[:6]
session_slug = f"{parent_prefix}{drive_base_dir.name or 'session'}_{path_hash}"
local_work_dir = Path('/content/decovo_work') / session_slug / 'results_decovo'
local_input_dir = Path('/content/decovo_work') / session_slug / 'input'
local_work_dir.mkdir(parents=True, exist_ok=True)
local_input_dir.mkdir(parents=True, exist_ok=True)

# Remove any stale files in local_input_dir that do not belong to this session
expected_names = {Path(f).name for f in spec.files}
for existing in local_input_dir.iterdir():
    if existing.is_file() and existing.name not in expected_names:
        existing.unlink()

# Stage input files onto local SSD for fast random page I/O
for src_file in spec.files:
    src_p = Path(src_file)
    dst_p = local_input_dir / src_p.name
    if not dst_p.exists() or dst_p.stat().st_size != src_p.stat().st_size:
        print(f'📥 Copying {src_p.name} to local Colab SSD...')
        shutil.copy2(src_p, dst_p)

# Restore checkpoints from Drive if resuming
if not start_fresh and drive_results_dir.exists():
    for ckpt in drive_results_dir.glob('*'):
        if ckpt.is_file() and (not ckpt.name.endswith('.mmap') or keep_memmap_on_drive):
            dst_ckpt = local_work_dir / ckpt.name
            if not dst_ckpt.exists():
                shutil.copy2(ckpt, dst_ckpt)

cfg = load_config(
    path_to_analyze=str(local_input_dir),
    results_folder=str(local_work_dir),
    frame_rate=float(effective_frame_rate),
    spatial_downsample=float(spatial_downsample),
    gSig=float(gSig),
)

session = preprocess(local_input_dir, config=cfg, force=bool(start_fresh))

# Mirror Stage A checkpoints to Google Drive
def sync_to_drive(include_mmap=False):
    copied = []
    for f in local_work_dir.glob('*'):
        if not f.is_file():
            continue
        if f.name.startswith('prep_') and f.name.endswith('.tif'):
            continue
        if f.name.endswith('.mmap') and not include_mmap:
            continue
        shutil.copy2(f, drive_results_dir / f.name)
        copied.append(f.name)
    return copied

synced = sync_to_drive(include_mmap=keep_memmap_on_drive)
print(f'✅ Stage A complete! Synced {len(synced)} checkpoint files to {drive_results_dir}')

In [ ]:
# @title 7. Stage B: Interactive Threshold Picker & Fast Extraction
# @markdown Drag `min_corr` and `min_pnr` to preview the seed mask, or click **Extract with these** to run CNMF-E + OASIS on the cached memmap.
# @markdown *(If you click 'Run all', an automatic extraction using the heuristic thresholds runs first so results are immediately ready.)*
from decovo.notebook_ui import threshold_picker

result = extract(session)
sync_to_drive(include_mmap=keep_memmap_on_drive)
print(f'✅ Initial auto-extraction complete: {result.n_neurons} accepted neurons (min_corr={result.min_corr_used:.3f}, min_pnr={result.min_pnr_used:.2f})')

def _on_reextract(new_res):
    global result
    result = new_res
    sync_to_drive(include_mmap=keep_memmap_on_drive)

_picker_ui, _picker_state = threshold_picker(session, on_extract=_on_reextract)

In [ ]:
# @title 8. Browse Extracted Neurons (Footprints, Raw & Deconvolved Traces)
from decovo.notebook_ui import neuron_browser

_browser_ui = neuron_browser(result)

In [ ]:
# @title 9. Save & Verify Outputs on Google Drive
copied_files = sync_to_drive(include_mmap=keep_memmap_on_drive)
print('=' * 68)
print(f'🎉 All results saved to Google Drive: {drive_results_dir}')
for fname in sorted(copied_files):
    fpath = drive_results_dir / fname
    size_mb = fpath.stat().st_size / (1024 * 1024)
    print(f'   • {fname:28s} ({size_mb:6.2f} MB)')
print('=' * 68)
print('👉 Download `analysis_results.hdf5` and open it in the CleanMiniscope Cell Curator GUI!')

In [ ]:
# @title 10. (Optional) Batch Process Multiple Drive Folders
run_batch_now = False # @param {type:"boolean"}
batch_folders_csv = '' # @param {type:"string"}

if run_batch_now and batch_folders_csv.strip():
    folders = [f.strip() for f in batch_folders_csv.split(',') if f.strip()]
    for idx, folder in enumerate(folders, 1):
        print(f'\n[{idx}/{len(folders)}] Processing {folder}...')
        f_path = Path(folder)
        b_spec = discover_inputs(f_path)
        b_fps = float(b_spec.fps_from_metadata) if (use_metadata_fps_if_present and b_spec.fps_from_metadata) else float(frame_rate)
        out_drive = (f_path.parent if f_path.is_file() else f_path) / 'results_decovo'
        out_drive.mkdir(parents=True, exist_ok=True)
        work = Path('/content/decovo_work') / f'batch_{idx}'
        b_cfg = load_config(
            path_to_analyze=str(f_path),
            results_folder=str(work),
            frame_rate=b_fps,
            spatial_downsample=float(spatial_downsample),
            gSig=float(gSig),
        )
        b_sess = preprocess(f_path, config=b_cfg, force=bool(start_fresh))
        b_res = extract(b_sess)
        for f in work.glob('*'):
            if f.is_file() and not f.name.endswith('.mmap') and not f.name.startswith('prep_'):
                shutil.copy2(f, out_drive / f.name)
        print(f'   ✅ Saved {b_res.n_neurons} neurons to {out_drive}')